# Paris Residential AVM — original exploratory notebook

This is the notebook the analysis was first written in: data quality audit,
cleaning, feature engineering, EDA, six models and diagnostics, in the order
they were actually done.

**Outputs are cleared.** The original run used a private listing extract that
cannot be redistributed, so nothing derived from it is committed here. To run
this notebook, generate the synthetic extract first:

```bash
python scripts/make_synthetic_data.py --rows 100000
```

For a shorter, refactored walkthrough that calls the `paris_avm` package
instead of repeating the code inline, see `01_analysis_walkthrough.ipynb`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)
import plotly.express as px

In [ ]:
DATE_COLS = ["start_date", "end_date", "publish_start_date"]


# 1. Data Loading & First Look

In [ ]:
filepath = "../data/listings_synthetic.csv"

df_raw = pd.read_csv(filepath, parse_dates=DATE_COLS)

In [ ]:
with pd.option_context('display.max_columns', None,
                       'display.max_colwidth', None,
                       'display.width', None):
    display(df_raw.head(10))

In [ ]:
print(f"Dataset Shape: {df_raw.shape[0]} rows and {df_raw.shape[1]} columns\n")

In [ ]:
df_raw.describe().T

In [ ]:
unique_df = pd.DataFrame({
    "Unique Values": df_raw.nunique(),
    "% Unique": (df_raw.nunique()/len(df_raw)*100).round(2)
})

display(unique_df.sort_values("Unique Values"))

# 2. Data Quality Audit

### 2.1 Missing Data

In [ ]:
missing = (df_raw.isna().mean() * 100).sort_values(ascending=False)
missing = missing[missing > 0]

fig, ax = plt.subplots(figsize=(10, 9))
sns.barplot(x=missing.values, y=missing.index, hue=missing.index, palette="flare", legend=False, ax=ax)
ax.set_xlabel("% missing"); ax.set_ylabel("")
ax.set_title("Missing data by column (raw dataset)")
for i, v in enumerate(missing.values):
    ax.text(v + 0.5, i, f"{v:.0f}%", va="center", fontsize=9)
plt.tight_layout()
plt.show()
missing

### 2.2 Duplicates

In [ ]:
dupe_mask = df_raw.duplicated(subset=["external_id"], keep=False)
print(f"{dupe_mask.sum()} rows share an external_id with another row "
      f"({dupe_mask.sum()//2} properties listed twice).")
df_raw[dupe_mask].sort_values("external_id")[
    ["external_id", "price", "start_date", "end_date", "publish_start_date"]
].head(8)

### 2.3 Geography quality

In [ ]:
print("Distinct spellings in `city`:", df_raw["city"].nunique())
df_raw["city"].value_counts().tail(10)

In [ ]:
valid_zip = df_raw["zipcode"].between(75001, 75020) #| (df_raw["zipcode"] == 75116)
print(f"{(~valid_zip).sum()} rows carry a zipcode that isn't a real Paris arrondissement:")
df_raw.loc[~valid_zip, ["zipcode", "city"]].value_counts()

### 2.4 Impossible values

In [ ]:
checks = pd.DataFrame({
    "min": df_raw[["price", "area", "floor", "floor_count", "room_count",
                    "bedroom_count", "build_year", "balcony_count", "terrace_count",
                    "bathroom_count", "toilet_count", "terrace_area", "balcony_area"]].min(),
    "max": df_raw[["price", "area", "floor", "floor_count", "room_count",
                    "bedroom_count", "build_year", "balcony_count", "terrace_count",
                    "bathroom_count", "toilet_count", "terrace_area", "balcony_area"]].max(),
})
checks

### 2.5 Property types

In [ ]:
df_raw["item_type"].value_counts()

### 2.6 Negative values

In [ ]:
count_cols = ['room_count', 'bedroom_count', 'bathroom_count', 'toilet_count',
              'balcony_count', 'terrace_count', 'parking_boxe_count',
              'parking_inside_count', 'floor', 'floor_count']

audit = pd.DataFrame({
    'min': df_raw[count_cols].min(),
    'max': df_raw[count_cols].max(),
    'n_negative': (df_raw[count_cols] < 0).sum(),
})
audit

### 2.7 build_year

In [ ]:
print(df_raw['build_year'].describe())
condition = (df_raw['build_year'] < 1600) | (df_raw['build_year'] > 2028)
print(f"build_year < 1600 or > 2028: {condition.sum()}")
display(df_raw.loc[condition, 'build_year'])

3.8 Start date

In [ ]:
oldest_start_date = df_raw['start_date'].min()
most_recent_start_date = df_raw['start_date'].max()

print(f"Oldest Start Date: {oldest_start_date}")
print(f"Most Recent Start Date: {most_recent_start_date}")

# 3. Data Cleaning


In [ ]:
# Work on a copy
df = df_raw.copy()

### 3.1 Standardize zipcodes

In [ ]:
# 1. Direct replacements for known mis-coded zipcodes
#    75116 -> 75016
#    75107 -> 75007, 75108 -> 75008
#    75000 -> 75003
zipcode_replacements = {75116: 75016, 75107: 75007, 75108: 75008, 75000: 75003}
df['zipcode'] = df['zipcode'].replace(zipcode_replacements)

# 2. Conditional replacements for 75550 and 75023
problematic_zipcodes = [75550, 75023]

def get_corrected_zipcode_for_group(group):
    if group['zipcode'].nunique() <= 1:
        return group['zipcode']
    valid_zipcodes = group[~group['zipcode'].isin(problematic_zipcodes)]['zipcode']
    if not valid_zipcodes.empty:
        most_common_valid_zip = valid_zipcodes.mode()[0]
        group['zipcode'] = group['zipcode'].apply(
            lambda x: most_common_valid_zip if x in problematic_zipcodes else x)
    return group['zipcode']

mask_for_grouping = df['address_id'].notna() & df['street_id'].notna()
df.loc[mask_for_grouping, 'zipcode'] = df[mask_for_grouping].groupby(
    ['address_id', 'street_id'], group_keys=False
).apply(get_corrected_zipcode_for_group)

# 3. Anything still not a valid Paris zipcode -> NaN
still_invalid = ~df['zipcode'].between(75001, 75020)
print(f"Setting {still_invalid.sum()} still-invalid zipcodes to NaN (row kept)")
df.loc[still_invalid, 'zipcode'] = np.nan
df = df.drop(columns=['address_id', 'street_id'])

print('Zipcode standardization complete. Shape:', df.shape)
print(f"Missing zipcodes remaining: {df['zipcode'].isna().sum()}")

### 3.2 Drop empty columns

In [ ]:
# Unnamed: 0                         -> spare row index, no info
# parcel_id                          -> 100% missing
# site_area, parking_garage_count    -> >99% missing
# city                               -> free-text, inconsistent -> superseded by zipcode (Section 2.3)
# iris_id, street_id, address_id     -> high-cardinality identifiers, not usable as model features directly
# client_id                          -> agency/client identifier, not a property attribute
drop_cols = ["Unnamed: 0", "parcel_id", "site_area", "parking_garage_count",
             "city", "iris_id", "street_id", "address_id", "client_id"]
drop_cols = [c for c in drop_cols if c in df.columns]
df = df.drop(columns=drop_cols)
print(f"Dropped {drop_cols}. Shape: {df.shape}")

### 3.3 Remove low-variability columns


In [ ]:
dominance = {}
for col in df.columns:
    vc = df[col].value_counts(normalize=True, dropna=True)
    if len(vc) > 0:
        dominance[col] = vc.iloc[0]

dominance = pd.Series(dominance).sort_values(ascending=False)
print("Share of rows taken by the single most common value, top 8:")
print(dominance.head(8))

near_constant_cols = dominance[dominance >= 0.99].index.tolist()
print(f"\nColumns flagged as near-constant (>=99% one value): {near_constant_cols}")

In [ ]:
if near_constant_cols:
    df = df.drop(columns=near_constant_cols)
    print(f"Dropped near-constant columns: {near_constant_cols}")
else:
    print("No near-constant columns found.")
print(f"Shape: {df.shape}")

### 3.4 Deduplicate re-listings

In [ ]:
n_before = len(df)
df = df.sort_values("end_date").drop_duplicates(subset=["external_id"], keep="last")
print(f"Removed {n_before - len(df)} duplicate re-listings. Shape: {df.shape}")

### 3.5 Clean item_type & item_subtype

In [ ]:
df['item_type'] = df['item_type'].str.replace('ITEM_TYPE.', '', regex=False)
df['item_subtype'] = df['item_subtype'].str.replace('ITEM_TYPE.', '', regex=False)

print('Unique item types:', df['item_type'].unique())
print('Unique item subtypes:', df['item_subtype'].unique())

### 3.6 filter Residential only

In [ ]:
before = len(df)
df = df[df['item_type'].isin(['APARTMENT', 'HOUSE'])]
print(f"Removed {before - len(df)} non-residential rows (parking, offices, storage...). Shape: {df.shape}")

### 3.7 Replace implausible values with NaN

In [ ]:
# Negative counts -> impossible
count_cols_to_fix = ['room_count', 'bedroom_count', 'bathroom_count', 'toilet_count',
                     'balcony_count', 'terrace_count', 'parking_boxe_count', 'parking_inside_count']
for col in count_cols_to_fix:
    df.loc[df[col] < 0, col] = np.nan

# Implausible maxima -> NaN
df.loc[df['room_count'] > 12, 'room_count'] = np.nan
df.loc[df['bedroom_count'] > 10, 'bedroom_count'] = np.nan
df.loc[df['bathroom_count'] > 6, 'bathroom_count'] = np.nan
df.loc[df['toilet_count'] > 6, 'toilet_count'] = np.nan
df.loc[(df['floor'] < -2) | (df['floor'] > 25), 'floor'] = np.nan
df.loc[(df['floor_count'] < 0) | (df['floor_count'] > 40), 'floor_count'] = np.nan
df.loc[(df['build_year'] < 1700) | (df['build_year'] > 2026), 'build_year'] = np.nan
df.loc[(df['terrace_area'] < 0) | (df['terrace_area'] > 200), 'terrace_area'] = np.nan
df.loc[(df['balcony_area'] < 0) | (df['balcony_area'] > 100), 'balcony_area'] = np.nan

df.loc[df['balcony_count'] > 8, 'balcony_count'] = np.nan
df.loc[df['terrace_count'] > 8, 'terrace_count'] = np.nan
df.loc[df['parking_boxe_count'] > 6, 'parking_boxe_count'] = np.nan
df.loc[df['parking_inside_count'] > 6, 'parking_inside_count'] = np.nan

print('Implausible values converted to NaN.')

### 3.8 Outlier removal (business rules on price)

In [ ]:
before = len(df)
price_ok = df["price"].between(20_000, 15_000_000)      # residential transaction range
area_ok  = df["area"].between(9, 400)                     # 9sqm = legal minimum habitable surface (decree 2002-120)
ppsqm    = df["price"] / df["area"]
ppsqm_ok = ppsqm.between(6_000, 18_000)                   # realistic 2025 Paris EUR/sqm band (8,631-14,832 by arrondissement)

keep = price_ok & area_ok & ppsqm_ok
print(f"Removing {(~keep).sum()} rows failing price/area/price-per-sqm bounds")
df = df[keep]
print(f"Shape after outlier filtering: {df.shape}  ({len(df)/len(df_raw):.1%} of raw data retained)")

In [ ]:
ppsqm

### 3.9 Structural missing values

In [ ]:
zero_fill_cols = [
    "balcony_count", "terrace_count", "terrace_area", "balcony_area",
    "parking_boxe_count", "parking_inside_count", "has_cellar", "has_passenger_lift",
]
zero_fill_cols = [c for c in zero_fill_cols if c in df.columns]
for col in zero_fill_cols:
    df[col] = df[col].fillna(0)
print("Structural zero-fill applied to:", zero_fill_cols)

### 3.10 Median imputation

In [ ]:
median_fill_cols = ["room_count", "bedroom_count", "bathroom_count", "toilet_count",
                    "floor", "floor_count", "build_year"]
median_fill_cols = [c for c in median_fill_cols if c in df.columns]

medians = df[median_fill_cols].median()
print("Medians used for imputation:")
print(medians)

for col in median_fill_cols:
    df[col] = df[col].fillna(medians[col])

print(f"\nFinal cleaned shape: {df.shape}  |  retained {len(df)/len(df_raw):.1%} of the original {len(df_raw):,} rows")
print(f"Missing values remaining: {df.isnull().sum().sum()}")

### 3.11 Energy certificate

A best (7) → G worst (1)

In [ ]:
energy_order = {
    'ENERGY.CERTIFICATE_EFFICIENCY_SCORE.A': 7,
    'ENERGY.CERTIFICATE_EFFICIENCY_SCORE.B': 6,
    'ENERGY.CERTIFICATE_EFFICIENCY_SCORE.C': 5,
    'ENERGY.CERTIFICATE_EFFICIENCY_SCORE.D': 4,
    'ENERGY.CERTIFICATE_EFFICIENCY_SCORE.E': 3,
    'ENERGY.CERTIFICATE_EFFICIENCY_SCORE.F': 2,
    'ENERGY.CERTIFICATE_EFFICIENCY_SCORE.G': 1,
}
df['energy_score_ordinal'] = df['energy_certificate_efficiency_score'].map(energy_order)
print(df['energy_score_ordinal'].value_counts(dropna=False).sort_index())

### 3.12 Start Date

In [ ]:
original_df_rows = df.shape[0]
df['start_date'] = pd.to_datetime(df['start_date'])
df_2025 = df[df['start_date'].dt.year == 2025]
print(f"Shape of data with start_date in 2025: {df_2025.shape}")

rows_removed = original_df_rows - df_2025.shape[0]
print(f"Number of rows removed: {rows_removed}")

### 3.13 Final cleaned dataset summary

In [ ]:
print(f"Raw shape:     {df_raw.shape}")
print(f"Cleaned shape: {df.shape}")
print(f"Retained:      {len(df)/len(df_raw):.1%} of original rows")
print(f"Missing values remaining: {df.isnull().sum().sum()}")
df.head()

# 4. Feature Engineering

### 4.1 Temporal features

In [ ]:
df["listing_year"] = df["publish_start_date"].dt.year
df["listing_month"] = df["publish_start_date"].dt.month
df["listing_quarter"] = df["publish_start_date"].dt.quarter
df["listing_dow"] = df["publish_start_date"].dt.dayofweek
df["days_on_market"] = (df["end_date"] - df["start_date"]).dt.days

### 4.2 Property age

In [ ]:
df["property_age"] = df["listing_year"] - df["build_year"]
df.loc[df["property_age"] < 0, "property_age"] = 0

### 4.3 Size & layout ratios

In [ ]:
df["area_per_room"] = df["area"] / df["room_count"].replace(0, np.nan)
df["area_per_bedroom"] = df["area"] / df["bedroom_count"].replace(0, np.nan)
df["outdoor_area"] = df["terrace_area"] + df["balcony_area"]

### 4.4 Parking

In [ ]:
parking_cols = [c for c in ["parking_boxe_count", "parking_inside_count"] if c in df.columns]
df["total_parking"] = df[parking_cols].sum(axis=1)

### 4.5 Position in building

In [ ]:
df["floor_ratio"] = df["floor"] / df["floor_count"].replace(0, np.nan)

### 4.6 Geography knowledge features


In [ ]:
df["arrondissement"] = df["zipcode"] - 75000
left_bank = {5, 6, 7, 13, 14, 15}
df["rive_gauche"] = df["arrondissement"].isin(left_bank).astype(int)
prime_west = {6, 7, 8, 16}
df["prime_arrondissement"] = df["arrondissement"].isin(prime_west).astype(int)

### 4.7 Target transformations

In [ ]:
df["log_price"] = np.log(df["price"])
df["log_area"] = np.log(df["area"])
df["price_per_sqm"] = df["price"] / df["area"]

### 4.8 Item_Type and Item_subtype

In [ ]:
# One-hot encode item_type and item_subtype
item_type_dummies = pd.get_dummies(df["item_type"], prefix="type", drop_first=True)
item_subtype_dummies = pd.get_dummies(df["item_subtype"], prefix="subtype", drop_first=True)

df = pd.concat([df, item_type_dummies, item_subtype_dummies], axis=1)

print(f"Added {item_type_dummies.shape[1]} item_type dummies: {item_type_dummies.columns.tolist()}")
print(f"Added {item_subtype_dummies.shape[1]} item_subtype dummies: {item_subtype_dummies.columns.tolist()}")

### 4.9 Feature engineering summary

In [ ]:
print(f"Engineered dataset: {df.shape}")
df[["area", "arrondissement", "property_age", "energy_score_ordinal", "rive_gauche",
    "prime_arrondissement", "log_price"]].head()

# 5. Exploratory Data Analysis (EDA)

### 5.1 Price distribution

In [ ]:
print(df["price"].describe().apply(lambda x: f"{x:,.0f}"))
print(f"\nMean/median ratio: {df['price'].mean()/df['price'].median():.2f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.histplot(df["price"]/1000, bins=80, ax=axes[0], color="#c0392b")
axes[0].set_xlabel("Price (k EUR)"); axes[0].set_title("Price distribution")
axes[0].xaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.0f}"))
sns.histplot(df["log_price"], bins=80, ax=axes[1], color="#2980b9", kde=True)
axes[1].set_xlabel("log(price)"); axes[1].set_title("log(Price) distribution")
plt.tight_layout(); plt.show()

### 5.2 Price vs Area

In [ ]:
df_plot = df.dropna(subset=['arrondissement'])
sample_df_plot = df_plot.sample(min(15000, len(df_plot)), random_state=42)
sample_df_plot['arrondissement_str'] = sample_df_plot['arrondissement'].astype(int).astype(str)
fig = px.scatter(
    sample_df_plot,
    x="area",
    y=sample_df_plot["price"]/1000, # Convert to k EUR
    color="arrondissement_str",
    title="Price vs Area (colored by arrondissement)",
    labels={
        "area": "Area (sqm)",
        "y": "Price (k EUR)",
        "arrondissement_str": "Arrondissement"
    },
    template="plotly_white",
    height=600
)
fig.update_layout(coloraxis_colorbar=dict(title="Arrondissement"))
fig.show()

### 5.3 Geography — price per sqm by arrondissement

In [ ]:
order = df.groupby("arrondissement")["price_per_sqm"].median().sort_values(ascending=False)
df_plot_boxplot = df.copy()
df_plot_boxplot['arrondissement_str'] = df_plot_boxplot['arrondissement'].astype(int).astype(str)

fig = px.box(
    df_plot_boxplot,
    x="arrondissement_str",
    y="price_per_sqm",
    category_orders={"arrondissement_str": [str(int(a)) for a in order.index]},
    title="Price per sqm by arrondissement",
    labels={
        "arrondissement_str": "Arrondissement",
        "price_per_sqm": "Price per sqm (EUR)"
    },
    color="arrondissement_str",
    boxmode="overlay",
    points=False,
    height=600
)

fig.update_traces(quartilemethod="exclusive")
fig.show()

print(f"Most expensive: arrondissement {int(order.index[0])} (median {order.iloc[0]:,.0f} EUR/sqm)")
print(f"Least expensive: arrondissement {int(order.index[-1])} (median {order.iloc[-1]:,.0f} EUR/sqm)")
print(f"Ratio: {order.iloc[0]/order.iloc[-1]:.1f}x")

### 5.4 Correlation overview

In [ ]:
num_cols = ["price", "area", "room_count", "bedroom_count", "bathroom_count",
            "floor", "floor_count", "property_age", "energy_score_ordinal",
            "total_parking", "outdoor_area", "arrondissement", 'days_on_market']
num_cols = [c for c in num_cols if c in df.columns]

# Build a log-price version for the Pearson matrix
df_log = df[num_cols].copy()
df_log["price"] = np.log1p(df_log["price"])
df_log = df_log.rename(columns={"price": "log_price"})

corr_pearson = df_log.corr(method="pearson", numeric_only=True)
corr_spearman = df[num_cols].corr(method="spearman", numeric_only=True)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

sns.heatmap(corr_pearson, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            ax=axes[0], vmin=-1, vmax=1, cbar_kws={"label": "Pearson r"})
axes[0].set_title("Pearson correlation — log(price)")

sns.heatmap(corr_spearman, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            ax=axes[1], vmin=-1, vmax=1, cbar_kws={"label": "Spearman ρ"})
axes[1].set_title("Spearman correlation — raw price")

plt.tight_layout()
plt.show()

### 5.5 Rooms & amenities

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
rc = df[df["room_count"] <= 8]
sns.boxplot(data=rc, x="room_count", y="price", hue="room_count", palette="flare", showfliers=False, legend=False, ax=ax)
ax.set_ylabel("Price (EUR)"); ax.set_xlabel("Number of rooms"); ax.set_title("Price by number of rooms")
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.0f}"))
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, col, label in zip(axes, ["has_passenger_lift", "has_cellar"], ["Passenger lift", "Cellar"]):
    tmp = df.groupby(col)["price_per_sqm"].median()
    sns.barplot(x=tmp.index.map({0: "No", 1: "Yes"}), y=tmp.values, hue=tmp.index,
                palette="crest", legend=False, ax=ax)
    ax.set_title(label); ax.set_ylabel("Median EUR/sqm"); ax.set_xlabel("")
plt.tight_layout(); plt.show()

### 5.6 Energy certificate (DPE) vs price

In [ ]:
sub = df.dropna(subset=["energy_score_ordinal"]).copy()
letters = list("ABCDEFG")
ordinal_to_letter = {7:"A",6:"B",5:"C",4:"D",3:"E",2:"F",1:"G"}
sub["energy_letter"] = sub["energy_score_ordinal"].map(ordinal_to_letter)
fig, ax = plt.subplots(figsize=(8, 5.5))
sns.boxplot(data=sub, x="energy_letter", y="price_per_sqm", order=letters,
            hue="energy_letter", palette="RdYlGn_r", legend=False, showfliers=False, ax=ax)
ax.set_title("Price per sqm by energy certificate (DPE)")
ax.set_xlabel("Energy certificate (A=best, G=worst)"); ax.set_ylabel("Price per sqm (EUR)")
plt.tight_layout(); plt.show()

### 5.7 Seasonality

In [ ]:
monthly = df.groupby("listing_month").agg(n=("price", "size"), med_ppsqm=("price_per_sqm", "median"))
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].bar(monthly.index, monthly["n"], color="#16a085")
axes[0].set_title("Listings published per month (2025)"); axes[0].set_xlabel("Month"); axes[0].set_ylabel("# listings")
axes[1].plot(monthly.index, monthly["med_ppsqm"], marker="o", color="#e67e22")
axes[1].set_title("Median price/sqm by month"); axes[1].set_xlabel("Month"); axes[1].set_ylabel("EUR/sqm")
plt.tight_layout(); plt.show()

# 6. Model Development

### 6.1 Feature selection for modeling

In [ ]:
# Candidate features. Excludes: identifiers, raw dates, derived target leakage
# (price_per_sqm, log_price, days_on_market), binary flags (dropped per design choice in Section 4),
# and log_area (redundant with area -- kept only in EDA, not as a model feature; avoids multicollinearity).
# Add the one-hot encoded item_type / item_subtype columns
feature_cols = [
    "area", "room_count", "bedroom_count", "bathroom_count", "toilet_count",
    "floor", "floor_count", "floor_ratio",
    "build_year", "property_age",
    "arrondissement", "rive_gauche", "prime_arrondissement",
    "has_passenger_lift", "has_cellar",
    "balcony_count", "terrace_count", "outdoor_area",
    "total_parking",
    "energy_score_ordinal",
    "area_per_room", "area_per_bedroom",
    "listing_month", "listing_quarter", "listing_dow",
]

feature_cols += item_type_dummies.columns.tolist()
feature_cols += item_subtype_dummies.columns.tolist()

feature_cols = [c for c in feature_cols if c in df.columns]

X = df[feature_cols].copy()
y = df["price"].copy()
y_log = df["log_price"].copy()

print(f"Number of features: {len(feature_cols)}")
print(feature_cols)
print(f"\nX shape: {X.shape}  |  y shape: {y.shape}")

### 6.2 Train-test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
_, _, y_train_log, y_test_log = train_test_split(X, y_log, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape[0]:,} rows")
print(f"Test set:     {X_test.shape[0]:,} rows")

### 6.3 Imputation & scaling

In [ ]:
imputer = SimpleImputer(strategy="median")
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=feature_cols, index=X_train.index)
X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=feature_cols, index=X_test.index)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imp)
X_test_scaled = scaler.transform(X_test_imp)

### 6.3 Baseline model: Linear Regression


In [ ]:
# ============================================================
# Option A — Linear Regression on RAW price
# ============================================================
baseline_raw_model = LinearRegression()
baseline_raw_model.fit(X_train_scaled, y_train)

pred_train_raw = baseline_raw_model.predict(X_train_scaled)
pred_test_raw = baseline_raw_model.predict(X_test_scaled)

baseline_raw_mae = mean_absolute_error(y_test, pred_test_raw)
baseline_raw_rmse = np.sqrt(mean_squared_error(y_test, pred_test_raw))
baseline_raw_r2 = r2_score(y_test, pred_test_raw)

print("Baseline (Linear Regression on RAW price) — Test set:")
print(f"  MAE:  EUR {baseline_raw_mae:,.0f}")
print(f"  RMSE: EUR {baseline_raw_rmse:,.0f}")
print(f"  R²:   {baseline_raw_r2:.4f}")

# ============================================================
# Option B — Linear Regression on LOG price
# ============================================================
baseline_model = LinearRegression()
baseline_model.fit(X_train_scaled, y_train_log)

pred_train_log = baseline_model.predict(X_train_scaled)
pred_test_log = baseline_model.predict(X_test_scaled)

log_lo, log_hi = y_train_log.min(), y_train_log.max()
pred_train_log = np.clip(pred_train_log, log_lo, log_hi)
pred_test_log = np.clip(pred_test_log, log_lo, log_hi)

pred_train = np.exp(pred_train_log)
pred_test = np.exp(pred_test_log)

baseline_test_mae = mean_absolute_error(y_test, pred_test)
baseline_test_rmse = np.sqrt(mean_squared_error(y_test, pred_test))
baseline_test_r2 = r2_score(y_test, pred_test)

print("\nBaseline (Linear Regression on LOG-price, back-transformed) — Test set:")
print(f"  MAE:  EUR {baseline_test_mae:,.0f}")
print(f"  RMSE: EUR {baseline_test_rmse:,.0f}")
print(f"  R²:   {baseline_test_r2:.4f}")

# ============================================================
# Comparison
# ============================================================
baseline_comparison = pd.DataFrame({
    "Approach": ["Raw price", "Log price "],
    "Test MAE (EUR)": [baseline_raw_mae, baseline_test_mae],
    "Test RMSE (EUR)": [baseline_raw_rmse, baseline_test_rmse],
    "Test R²": [baseline_raw_r2, baseline_test_r2],
})
print("\n" + "="*60)
print("Raw price vs. log-price target comparison")
print("="*60)
print(baseline_comparison.to_string(index=False))

cv_scores = cross_val_score(baseline_model, X_train_scaled, y_train_log, cv=5, scoring="r2")
print(f"\n5-fold CV R² (log-price model): {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

### 6.4 Advanced models

In [ ]:
results = {}

def evaluate(name, model, X_tr, y_tr, X_te):
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)
    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)
    results[name] = {"model": model, "MAE": mae, "RMSE": rmse, "R2": r2, "pred": pred}
    print(f"{name:20s} | MAE: EUR {mae:>10,.0f} | RMSE: EUR {rmse:>10,.0f} | R²: {r2:.4f}")

results["Linear Regression"] = {"model": baseline_raw_model, "MAE": baseline_raw_mae,
                                 "RMSE": baseline_raw_rmse, "R2": baseline_raw_r2, "pred": pred_test_raw}

evaluate("Ridge", Ridge(alpha=1.0), X_train_scaled, y_train, X_test_scaled)
evaluate("Lasso", Lasso(alpha=0.001, max_iter=50000), X_train_scaled, y_train, X_test_scaled)
evaluate("Random Forest",
         RandomForestRegressor(n_estimators=300, max_depth=18, min_samples_leaf=3,
                                random_state=42, n_jobs=-1),
         X_train_imp, y_train, X_test_imp)
evaluate("Gradient Boosting",
         GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=42),
         X_train_imp, y_train, X_test_imp)
evaluate("XGBoost",
         xgb.XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=6,
                           subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=0),
         X_train_imp, y_train, X_test_imp)

### 6.5 Model comparison

In [ ]:
comparison = pd.DataFrame({
    "Model": list(results.keys()),
    "Test MAE (EUR)": [results[m]["MAE"] for m in results],
    "Test RMSE (EUR)": [results[m]["RMSE"] for m in results],
    "Test R²": [results[m]["R2"] for m in results],
}).sort_values("Test R²", ascending=False).reset_index(drop=True)

print(comparison.to_string(index=False))

best_model_name = comparison.loc[0, "Model"]
best_model = results[best_model_name]["model"]
y_pred_best = results[best_model_name]["pred"]
print(f"\nBest model: {best_model_name}  (Test R² = {comparison.loc[0, 'Test R²']:.4f})")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ["gold" if m == best_model_name else "steelblue" for m in comparison["Model"]]

axes[0].bar(comparison["Model"], comparison["Test MAE (EUR)"], color=colors, edgecolor="black")
axes[0].set_title("Test MAE"); axes[0].tick_params(axis="x", rotation=45)

axes[1].bar(comparison["Model"], comparison["Test RMSE (EUR)"], color=colors, edgecolor="black")
axes[1].set_title("Test RMSE"); axes[1].tick_params(axis="x", rotation=45)

axes[2].bar(comparison["Model"], comparison["Test R²"], color=colors, edgecolor="black")
axes[2].set_title("Test R²"); axes[2].tick_params(axis="x", rotation=45)

plt.tight_layout(); plt.show()

# 7. Model Evaluation & Diagnostics

### 7.1 Predictions vs Actual

In [ ]:
residuals = y_test.values - y_pred_best

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(y_test, y_pred_best, alpha=0.4, s=12, color="steelblue")
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--", lw=2)
axes[0].set_xlabel("Actual Price (EUR)"); axes[0].set_ylabel("Predicted Price (EUR)")
axes[0].set_title(f"{best_model_name} — Actual vs Predicted")

axes[1].scatter(y_pred_best, residuals, alpha=0.4, s=12, color="coral")
axes[1].axhline(0, color="r", linestyle="--", lw=2)
axes[1].set_xlabel("Predicted Price (EUR)"); axes[1].set_ylabel("Residual (EUR)")
axes[1].set_title(f"{best_model_name} — Residuals")
plt.tight_layout(); plt.show()

### 7.2 Residual distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(residuals, bins=60, ax=ax, color="skyblue")
ax.axvline(0, color="r", linestyle="--", label=f"Mean: EUR {residuals.mean():,.0f}")
ax.set_xlabel("Residual (EUR)"); ax.set_title("Residual distribution")
ax.legend(); plt.tight_layout(); plt.show()

### 7.3 Error by price segment

In [ ]:
error_df = pd.DataFrame({"actual": y_test.values, "pred": y_pred_best})
error_df["abs_error"] = (error_df["actual"] - error_df["pred"]).abs()
error_df["pct_error"] = error_df["abs_error"] / error_df["actual"] * 100
error_df["price_bucket"] = pd.qcut(error_df["actual"], 5,
                                    labels=["Q1 (cheapest)", "Q2", "Q3", "Q4", "Q5 (priciest)"])

seg = error_df.groupby("price_bucket").agg(
    n=("actual", "size"),
    mean_price=("actual", "mean"),
    MAE=("abs_error", "mean"),
    MAPE=("pct_error", "mean"),
)
seg

### 7.4 Feature importance

In [ ]:
if hasattr(best_model, "feature_importances_"):
    importance = pd.DataFrame({
        "Feature": feature_cols,
        "Importance": best_model.feature_importances_
    }).sort_values("Importance", ascending=False)
else:
    importance = pd.DataFrame({
        "Feature": feature_cols,
        "Importance": np.abs(best_model.coef_)
    }).sort_values("Importance", ascending=False)

print(importance.to_string(index=False))

In [ ]:
top_n = importance.head(15)
fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(range(len(top_n)), top_n["Importance"].values, color="steelblue", edgecolor="black")
ax.set_yticks(range(len(top_n))); ax.set_yticklabels(top_n["Feature"].values)
ax.invert_yaxis()
ax.set_xlabel("Importance"); ax.set_title(f"Top 15 features — {best_model_name}")
plt.tight_layout(); plt.show()

In [ ]:
geo_check = error_df.copy()
geo_check["arrondissement"] = X_test["arrondissement"].values
geo_actual = geo_check.groupby("arrondissement")["actual"].median()
geo_pred = geo_check.groupby("arrondissement")["pred"].median()

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(geo_actual.index, geo_actual.values, marker="o", label="Actual median price")
ax.plot(geo_pred.index, geo_pred.values, marker="s", label="Predicted median price")
ax.set_xlabel("Arrondissement"); ax.set_ylabel("Median Price (EUR)")
ax.set_title("Actual vs Predicted median price by arrondissement (test set)")
ax.legend(); plt.tight_layout(); plt.show()